In [3]:
import sys
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
from pathlib import Path
from unidecode import unidecode
from tqdm import tqdm

In [4]:
SPOTIFY_CHUNKSIZE = 200000
GRAMMY_IN = Path("grammy_winners.csv")
SPOTIFY_IN = Path("spotify_data.csv")
OUT_DIR = Path("cleaned")
OUT_DIR.mkdir(exist_ok=True)
GRAMMY_OUT = OUT_DIR / "grammy_nominees_clean.csv"
SPOTIFY_OUT = OUT_DIR / "spotify_clean.csv"
YEAR_MIN, YEAR_MAX = 2000, 2023

In [6]:
def normalize_text(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s)
    s = unidecode(s)
    s = s.lower().strip()
    # remove content in parentheses/brackets/quotes (these often include versions/feat)
    s = re.sub(r"[\(\[\{].*?[\)\]\}]", " ", s)
    # remove 'feat' and everything following common patterns (but keep primary artist)
    s = re.sub(r"\b(feat|ft|featuring|featuring:|featuring-)\b.*", " ", s)
    # remove punctuation except letters, numbers, &, + and spaces
    s = re.sub(r"[^a-z0-9&\+\s]", " ", s)
    # collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

def simple_track_normalize(s: str) -> str:
    s = normalize_text(s)
    # remove trailing markers like 'remix', 'live', 'edit'
    s = re.sub(r"\b(remix|live|edit|version|explicit|radio edit|acoustic)\b.*$", "", s).strip()
    return s

def _is_unknown_artist(val):
    if pd.isna(val):
        return True
    s = str(val).strip().lower()
    return s == "" or s == "unknown"

def clean_grammy(inpath: Path, outpath: Path):
    df = pd.read_csv(inpath)

    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    if "Work" in df.columns:
        df = df.dropna(subset=["Work"])

    if "Year" in df.columns:
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
        df = df[(df["Year"] >= YEAR_MIN) & (df["Year"] <= YEAR_MAX)].copy()
        df["Year"] = df["Year"].astype(int)

    if "Nominee" in df.columns:
        df["Nominee"] = df["Nominee"].fillna("unknown")

    if "Nominee" in df.columns:
        unknown_mask = df["Nominee"].astype(str).apply(_is_unknown_artist)
        removed = int(unknown_mask.sum())
        df = df[~unknown_mask].copy()
        print(f"Removed {removed} Grammy rows with unknown/empty nominee (filtered to {YEAR_MIN}-{YEAR_MAX}).")

    if "Nominee" in df.columns:
        df["artist_norm"] = df["Nominee"].astype(str).apply(normalize_text)
    if "Work" in df.columns:
        df["track_norm"] = df["Work"].astype(str).apply(simple_track_normalize)

    dedupe_cols = [c for c in ["Year", "Award Name", "Nominee", "Work"] if c in df.columns]
    if dedupe_cols:
        before_dup = len(df)
        df = df.drop_duplicates(subset=dedupe_cols)
        print(f"Dropped {before_dup - len(df)} duplicate Grammy rows based on {dedupe_cols}.")

    df.to_csv(outpath, index=False)
    print(f"Grammy cleaned: {df.shape} -> saved to {outpath}")
    return df

def clean_spotify(inpath: Path, outpath: Path, chunksize=SPOTIFY_CHUNKSIZE):
    first_chunk = True
    dropped_rows_total = 0
    total_written = 0

    for chunk in tqdm(pd.read_csv(inpath, chunksize=chunksize), desc="Processing Spotify chunks"):
        if "Unnamed: 0" in chunk.columns:
            chunk = chunk.drop(columns=["Unnamed: 0"])

        if "year" in chunk.columns:
            chunk["year"] = pd.to_numeric(chunk["year"], errors="coerce")
            chunk = chunk[(chunk["year"] >= YEAR_MIN) & (chunk["year"] <= YEAR_MAX)]

        before = len(chunk)
        chunk = chunk.dropna(subset=["artist_name", "track_name"])
        dropped_rows_total += (before - len(chunk))

        chunk["artist_name_norm"] = chunk["artist_name"].astype(str).apply(normalize_text)
        chunk["track_name_norm"] = chunk["track_name"].astype(str).apply(simple_track_normalize)

        numeric_cols = ["popularity", "danceability", "energy", "loudness", "speechiness",
                        "acousticness", "instrumentalness", "liveness", "valence", "tempo", "duration_ms", "year"]
        for nc in numeric_cols:
            if nc in chunk.columns:
                chunk[nc] = pd.to_numeric(chunk[nc], errors="coerce")

        if "track_id" in chunk.columns:
            chunk = chunk.drop_duplicates(subset=["track_id"])
        else:
            chunk = chunk.drop_duplicates(subset=["artist_name_norm", "track_name_norm"])

        if first_chunk:
            chunk.to_csv(outpath, index=False, mode="w")
            first_chunk = False
        else:
            chunk.to_csv(outpath, index=False, mode="a", header=False)

        total_written += len(chunk)

    print(f"Spotify cleaned: total written rows approx {total_written}, total dropped rows in-chunks {dropped_rows_total}. Saved to {outpath}")

    df_sample = pd.read_csv(outpath)
    return df_sample

In [7]:
grammy_preview = pd.read_csv(GRAMMY_IN, nrows=10)
spotify_preview = pd.read_csv(SPOTIFY_IN, nrows=10)
print("GRAMMY PREVIEW COLUMNS:", grammy_preview.columns.tolist())
print("SPOTIFY PREVIEW COLUMNS:", spotify_preview.columns.tolist())

# Run cleaning
grammy_clean = clean_grammy(GRAMMY_IN, GRAMMY_OUT)
spotify_clean = clean_spotify(SPOTIFY_IN, SPOTIFY_OUT)

print("\nSample of cleaned spotify file:")
print(spotify_clean.head().to_string())

print("\nDone. Cleaned files saved to:", OUT_DIR)

GRAMMY PREVIEW COLUMNS: ['Unnamed: 0', 'Year', 'Ceremony', 'Award ID', 'Award Type', 'Award Name', 'Work', 'Nominee', 'Winner']
SPOTIFY PREVIEW COLUMNS: ['Unnamed: 0', 'artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature']
Removed 0 Grammy rows with unknown/empty nominee (filtered to 2000-2023).
Dropped 0 duplicate Grammy rows based on ['Year', 'Award Name', 'Nominee', 'Work'].
Grammy cleaned: (11095, 10) -> saved to cleaned/grammy_nominees_clean.csv


Processing Spotify chunks: 6it [00:39,  6.51s/it]


Spotify cleaned: total written rows approx 1159748, total dropped rows in-chunks 16. Saved to cleaned/spotify_clean.csv

Sample of cleaned spotify file:
     artist_name        track_name                track_id  popularity  year     genre  danceability  energy  key  loudness  mode  speechiness  acousticness  instrumentalness  liveness  valence    tempo  duration_ms  time_signature artist_name_norm   track_name_norm
0     Jason Mraz   I Won't Give Up  53QF56cjZA9RTuuMZDrSA6          68  2012  acoustic         0.483   0.303    4   -10.058     1       0.0429        0.6940          0.000000    0.1150    0.139  133.406       240166               3       jason mraz   i won t give up
1     Jason Mraz  93 Million Miles  1s8tP3jP4GZcyHDsjvw218          50  2012  acoustic         0.572   0.454    3   -10.286     1       0.0258        0.4770          0.000014    0.0974    0.515  140.182       216387               4       jason mraz  93 million miles
2  Joshua Hyslop  Do Not Let Me Go  7BRCa8MPiy

In [8]:
grammy_clean["combined_norm"] = (
    grammy_clean["artist_norm"].astype(str).str.strip()
    + " - " +
    grammy_clean["track_norm"].astype(str).str.strip()
)
spotify_clean["combined_norm"] = (
    spotify_clean["artist_name_norm"].astype(str).str.strip()
    + " - " +
    spotify_clean["track_name_norm"].astype(str).str.strip()
)

In [9]:
grammy_clean["combined_norm"] = grammy_clean["combined_norm"].astype(str).str.strip()
spotify_clean["combined_norm"] = spotify_clean["combined_norm"].astype(str).str.strip()

In [10]:
grammy = grammy_clean[grammy_clean["combined_norm"] != ""]
spotify = spotify_clean[spotify_clean["combined_norm"] != ""]

grammy_keys = set(grammy["combined_norm"])
spotify_keys = set(spotify["combined_norm"])

common_keys = grammy_keys.intersection(spotify_keys)

print("Total Grammy rows:", len(grammy))
print("Total Spotify rows:", len(spotify))
print("Total matching combined_norm keys:", len(common_keys))

print("\nExample matched keys:")
print(list(common_keys)[:10])

matched_grammy = grammy[grammy["combined_norm"].isin(common_keys)]
matched_spotify = spotify[spotify["combined_norm"].isin(common_keys)]

print("\nMatched Grammy rows:", matched_grammy.shape)
print("Matched Spotify rows:", matched_spotify.shape)

Total Grammy rows: 11095
Total Spotify rows: 1159748
Total matching combined_norm keys: 2103

Example matched keys:
['mary j blige - just fine', 'killswitch engage - the end of heartache', 'deadmau5 - while', 'gov t mule - sco mule', 'prince - 3121', 'foo fighters - times like these', 'corinne bailey rae - put your records on', 'alejandro sanz - el tren de los momentos', 'gojira - magma', 'slipknot - my plague']

Matched Grammy rows: (2327, 11)
Matched Spotify rows: (2627, 22)


In [11]:
merged = pd.merge(
    grammy,
    spotify,
    on="combined_norm",
    how="inner",
    suffixes=("_grammy", "_spotify")
)

print("Merged rows:", len(merged))
merged.columns.tolist()

Merged rows: 2954


['Year',
 'Ceremony',
 'Award ID',
 'Award Type',
 'Award Name',
 'Work',
 'Nominee',
 'Winner',
 'artist_norm',
 'track_norm',
 'combined_norm',
 'artist_name',
 'track_name',
 'track_id',
 'popularity',
 'year',
 'genre',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'duration_ms',
 'time_signature',
 'artist_name_norm',
 'track_name_norm']

In [12]:
columns_to_drop = [
    # redundant normalization columns
    "artist_norm",
    "track_norm",
    "artist_name_norm",
    "track_name_norm",

    # Spotify raw metadata
    "artist_name",
    "track_name",
    "track_id",
    "key",
    "time_signature",
    "year",

    # Grammy raw metadata
    "Award ID",
    "Award Type"
]

In [13]:
cols_existing = [c for c in columns_to_drop if c in merged.columns]
merged = merged.drop(columns=cols_existing)

print("Columns after cleanup:", merged.columns.tolist())

Columns after cleanup: ['Year', 'Ceremony', 'Award Name', 'Work', 'Nominee', 'Winner', 'combined_norm', 'popularity', 'genre', 'danceability', 'energy', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']


In [14]:
merged.to_csv("cleaned/grammy_spotify_merged.csv", index=False)